🧩 Bloque 1 – Importación de librerías

En este bloque se importan las librerías necesarias para:
- Manipular archivos Excel y datos con pandas y numpy.
- Conectarse a la base de datos SQL Server mediante SQLAlchemy.
- Trabajar con fechas, rutas, archivos y expresiones regulares.

Estas herramientas permiten la automatización completa del proceso de lectura, limpieza y carga de datos en SQL Server.

In [ ]:
import pandas as pd

import sys
import importlib
import json
import os
from os import path
from pathlib import Path

# --- Resolver rutas ---
# notebook: .../traspaso-innominados/new_source/new_notebooks/banigualdad/
# queremos:  .../traspaso-innominados/functions
#            .../traspaso-innominados/config
PROJECT_ROOT = Path(os.environ["PROJECT_ROOT"]).resolve()
FUNCTIONS_DIR = PROJECT_ROOT / "functions"
CONFIG_DIR = PROJECT_ROOT / "config"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FUNCTIONS_DIR:", FUNCTIONS_DIR, "exists:", FUNCTIONS_DIR.exists())
print("CONFIG_DIR:", CONFIG_DIR, "exists:", CONFIG_DIR.exists())

# --- Insertar al sys.path (sin duplicar) ---
functions_path = str(FUNCTIONS_DIR)
if functions_path not in sys.path:
    sys.path.insert(0, functions_path)

# --- Importar (o recargar si ya estaba importado) ---
# Si tu archivo se llama functions.py dentro de FUNCTIONS_DIR, esto funciona.
# Si es paquete "functions/" con __init__.py, esto también funciona.
module_name = "functions"

try:
    if module_name in sys.modules:
        fun = importlib.reload(sys.modules[module_name])
    else:
        fun = importlib.import_module(module_name)

    print("Módulo importado desde:", getattr(fun, "__file__", "<sin __file__>"))

except Exception as e:
    # Mensaje más útil al depurar imports
    raise ImportError(
        f"No se pudo importar el módulo '{module_name}'.\n"
        f"Revisa que exista en: {FUNCTIONS_DIR}\n"
        f"sys.path{sys.path[0] if sys.path else '<vacío>'}\n"
        f"Detalle: {e}"
    )

# --- Cargar config JSON ---
config_file = path.join(CONFIG_DIR, "config_marsh.json")

try:
    with open(config_file, "r", encoding="utf-8") as file:
        data = json.load(file)

    # 'data' es un dict, no tiene __file__. Imprime la ruta del archivo real.
    print("Config cargada desde:", config_file)
    print("Claves disponibles:", list(data.keys()) if isinstance(data, dict) else type(data))

except FileNotFoundError:
    raise FileNotFoundError(
        f"No se encontró el archivo de configuración:\n{config_file}\n"
        f"¿Existe CONFIG_DIR?: {CONFIG_DIR.exists()}"
    )
except json.JSONDecodeError as e:
    raise ValueError(f"Error: JSON inválido en {config_file}: {e}")
except Exception as e:
    raise RuntimeError(f"Error inesperado leyendo {config_file}: {e}")

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
FUNCTIONS_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\functions exists: True
CONFIG_DIR: C:\Users\aepmvalenzuela\traspaso-innominados\config exists: True
Módulo importado desde: C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py
Config cargada desde: C:\Users\aepmvalenzuela\traspaso-innominados\config\config_marsh.json
Claves disponibles: ['tablas_remotas', 'server_config']


🗄️ Bloque 2 – Conexión a SQL Server

En este bloque se configura la conexión al servidor SQL Server utilizando SQLAlchemy y el driver ODBC 17.

Se definen variables de conexión (server, database, schema, table).

Se crea el objeto engine, que será utilizado para ejecutar consultas e insertar datos.

Los parámetros fast_executemany=True y pool_pre_ping=True mejoran el rendimiento y evitan errores de conexión inactiva.

In [6]:
# --- Parámetros ---
server = data["server_config"]["server"]
database = data["server_config"]["database"]
schema = data["server_config"]["schema"]

# --- Construir engine + obtener connection_string pyodbc ---
# Usa tus funciones: build_sqlserver_engine (y opcionalmente diagnósticos)

driver = "ODBC Driver 17 for SQL Server"  # o "ODBC Driver 18 for SQL Server"

# 1) ODBC connection string (pyodbc) -> para query_to_df / df_to_db / exec_sql
connection_string = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Connection Timeout=5;"
)

# Si tu entorno requiere TLS/cert (muy común con Driver 18), descomenta:
# connection_string += "Encrypt=yes;TrustServerCertificate=yes;"

# 2) SQLAlchemy engine -> usando tu helper (hace test SELECT 1 y crea el engine)
engine = fun.build_sqlserver_engine(
    server=server,
    database=database,
    driver=driver,
    trusted_connection=True,
    timeout=5,
    # encrypt=True, trust_server_certificate=True,  # si lo necesitas
    fast_executemany=True,
    pool_pre_ping=True,
    on_fail="raise",  # "warn" si prefieres que no explote y solo imprima diagnóstico
)

📁 Bloque 3 – Configuración de ruta y hoja preferida

En este bloque se define la carpeta donde se buscarán los archivos Excel y se establecen las configuraciones básicas:
- RUTA_ARCHIVOS: ubicación de los archivos.
- EXTS: extensiones válidas (por defecto .xlsx).
- PREFERRED_SHEETS: hojas de cálculo preferidas que el script buscará automáticamente.

Esto permite encontrar y seleccionar automáticamente el archivo más reciente y la hoja de trabajo correcta.

In [7]:
RUTA_ARCHIVOS = (PROJECT_ROOT / "data" / "CCLA" / "MARSH").resolve()


print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUTA_ARCHIVOS:", RUTA_ARCHIVOS, "exists:", RUTA_ARCHIVOS.exists())

#RUTA_ARCHIVOS = Path(r"C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\MARSH")
EXTS = (".xlsx",)  # Extensiones de archivo permitidas
#Se recomienda transformar el archivo ya que habitualmente llega de forma xlsb, transformarlo a xlsx

PREFERRED_SHEETS = ["base", "Base cesantia SURA Stock", "Base"]
FALLBACK_SHEET_INDEX = 0

PROJECT_ROOT: C:\Users\aepmvalenzuela\traspaso-innominados
RUTA_ARCHIVOS: C:\Users\aepmvalenzuela\traspaso-innominados\data\CCLA\MARSH exists: True


📄 Bloque 5 – Selección del archivo más reciente y hoja de trabajo

En este bloque se identifica el archivo Excel más reciente dentro de la carpeta configurada.
- Se ordenan los archivos por fecha de modificación.
- Se selecciona la hoja de trabajo más adecuada según los nombres definidos.
- Si el archivo está bloqueado por OneDrive, se crea una copia temporal para poder leerlo.

Esto asegura que el proceso siempre trabaje con el archivo más actualizado.

In [8]:
info = fun.get_latest_excel_and_sheet(
    RUTA_ARCHIVOS,
    EXTS,
    PREFERRED_SHEETS,
    fallback_sheet_index=FALLBACK_SHEET_INDEX,
    recursive=False,
    engine="openpyxl",
)

archivo_origen   = info["archivo_origen"]
excel_path_used  = info["excel_path_used"]
_tmp_copy_path   = info["tmp_copy_path"]
target           = info["target_sheet"]
sheet_names      = info["sheet_names"]

# (Opcional) leer hoja
df_opexcel = fun.read_excel_safe_no_header(excel_path_used, target)
fun.pretty_table(df_opexcel,n=5)

Archivo más reciente: SC202511_Ok Sura.xlsx  |  2026-01-19 16:45:02
Hojas: ['base', 'Resumen']
Hoja objetivo: base


📊 Bloque 6 – Lectura segura del Excel y limpieza de encabezados

En este bloque se realiza la lectura del archivo Excel seleccionado y la limpieza inicial de los datos:
- Se utiliza pd.read_excel(..., dtype=str) para mantener el control total de los tipos de datos.
- Se eliminan valores vacíos y se normalizan los encabezados con la función fun.normalize_name.
- Se imprime una vista previa del DataFrame resultante.

El resultado es un DataFrame uniforme y listo para su mapeo.

In [9]:
io = _tmp_copy_path if _tmp_copy_path else archivo_origen
df_raw = fun.read_excel_safe(io, target)

for c in df_raw.columns:
    if df_raw[c].dtype == object:
        df_raw[c] = df_raw[c].astype(str).str.strip().replace({"None": "", "nan": "", "NaN": ""})
df_raw.columns = [fun.normalize_name(c) for c in df_raw.columns]
print("Encabezados normalizados:", list(df_raw.columns))
fun.pretty_table(df_raw, n=5)

Encabezados normalizados: ['foliocredito', 'rutafiliado', 'dvafiliado', 'nombreafiliado', 'plazo', 'montobruto', 'fecotorgamiento', 'fechaprimervto', 'fechaultimovto', 'valorcuota', 'fechaprima', 'prima', 'desgravamen', 'fechadefuncion', 'origendefuncion', 'producto', 'folioorigen', 'fechaorigen', 'tasaorigen', 'corredora', 'unnamed_20', 'fecha_de_venta', 'fecha_de_baja', 'poliza', 'tasa', 'prima_bruta_mensual', 'iva', 'prima_neta', 'diferencia_ccla']


🧱 Bloque 7 – Mapeo de columnas origen → destino

En este bloque se mapean las columnas originales del Excel hacia las columnas estándar del sistema:
- Se utiliza la función fun.pick() para tolerar cambios o variaciones en los encabezados.
- Se construye el DataFrame df_can con todas las columnas requeridas.
- Se agrega la columna Nombre_de_archivo para identificar la fuente del dato.

Este paso genera una estructura homogénea y compatible con la base de datos.

In [10]:
# Origen → Destino
foliocredito        = fun.pick(df_raw, "n_operacion", "no_operacion", "noperacion", "operacion", "folio", "foliocredito")
rutafiliado         = fun.pick(df_raw, "afirut", "rut_afiliado", "rut", "rutafiliado")
dvafiliado          = fun.pick(df_raw, "afirutdv", "dv_afiliado", "dv", "dvafiliado")
NombreAfiliado      = fun.pick(df_raw, "afinom", "nombre_afiliado", "nombre", "nombreafiliado")
Plazo               = fun.pick(df_raw, "crecuotot", "plazo")
MontoBruto          = fun.pick(df_raw, "cresolmon", "monto_bruto", "monto", "montobruto")
fecotorgamiento     = fun.pick(df_raw, "fecotorgamiento", "fecha_otorgamiento", "fec_otorgamiento")
fechaPrimerVto      = fun.pick(df_raw, "fecinicob", "fecha_primer_vto", "fec_ini_cob", "fechaprimervto")
FechaUltimoVto      = fun.pick(df_raw, "fectercob", "fecha_ultimo_vto", "fec_ter_cob", "fechaultimovto")
ValorCuota          = fun.pick(df_raw, "crecuomon", "valor_cuota", "valorcuota")
FechaPrima          = fun.pick(df_raw, "fechaprima", "fecha_prima")
Prima               = fun.pick(df_raw, "prima_cliente", "prima_bruta", "prima_cliente_" , "prima")
Desgravamen         = fun.pick(df_raw, "prima_desgravamen", "desgravamen")
FechaDefuncion      = fun.pick(df_raw, "fecdefuncion", "fecha_defuncion", "fechadefuncion")
OriginDefuncion     = fun.pick(df_raw, "origen_defuncion", "origin_defuncion", "origendefuncion")
Producto            = fun.pick(df_raw, "producto")
FolioOrigen         = fun.pick(df_raw, "folio_original", "folio_origen", "folioorigen")
FechaOrigen         = fun.pick(df_raw, "fecha_otorgamiento_folio_original", "fecha_origen", "fechaorigen")
TasaOrigen          = fun.pick(df_raw, "tasa_interes", "tasaorigen")
Corredora           = fun.pick(df_raw, "corredora")
Fecha_de_venta      = fun.pick(df_raw, "fecha_de_venta", "fechadeventa", "fecha_venta")
Fecha_de_baja       = fun.pick(df_raw, "fecha_de_baja", "fechadebaja", "fecha_baja")
Poliza              = fun.pick(df_raw, "poliza", "póliza")
Tasa                = fun.pick(df_raw, "tasa_interes", "tasa")
PrimaBruta          = fun.pick(df_raw, "prima_bruta_mensual", "prima_bruta")
Iva                 = fun.pick(df_raw, "iva")
PrimaNeta           = fun.pick(df_raw, "primaneta", "prima_neta")
Diferencia_CCLA     = fun.pick(df_raw, "diferencia_ccla", "diferenciaccla")


df_can = pd.DataFrame({
    "foliocredito": foliocredito,
    "rutafiliado": rutafiliado,
    "dvafiliado": dvafiliado,
    "NombreAfiliado": NombreAfiliado,
    "Plazo": Plazo,
    "MontoBruto": MontoBruto,
    "fecotorgamiento": fecotorgamiento,
    "fechaPrimerVto": fechaPrimerVto,
    "FechaUltimoVto": FechaUltimoVto,
    "ValorCuota": ValorCuota,
    "FechaPrima": FechaPrima,
    "Prima": Prima,
    "Desgravamen": Desgravamen,
    "FechaDefuncion": FechaDefuncion,
    "OrigenDefuncion": OriginDefuncion,
    "Producto": Producto,
    "FolioOrigen": FolioOrigen,
    "FechaOrigen": FechaOrigen,
    "TasaOrigen": TasaOrigen,
    "Corredora": Corredora,
    "FECHA_DE_VENTA": Fecha_de_venta,
    "FECHA_DE_BAJA": Fecha_de_baja,
    "Poliza": Poliza,
    "Tasa": Tasa,
    "Prima_Bruta_mensual": PrimaBruta,
    "IVA": Iva,
    "Prima_Neta": PrimaNeta,
    "Diferencia_CCLA": Diferencia_CCLA,
    "Nombre_de_archivo": archivo_origen.name,
})

fun.pretty_table(df_can, n=5)

🕓 Bloque 8 – Extracción de período y nombre canónico

En este bloque se extrae el período (YYYYMM) desde el nombre del archivo y se genera un nombre canónico estandarizado:
- La función _extract_yyyymm_from_name() detecta el mes y año.
- canonicalizar_planes_safe() asegura que el proceso no se detenga si no se encuentra el período.
- Se reemplaza Nombre_de_archivo por el nombre canónico (SC_MARSHYYYYMM.xlsx).

Esto permite identificar los archivos de manera única y ordenada dentro de la base de datos.

In [11]:
nombre_canonico = fun.canonicalizar_planes(archivo_origen.name)
print("Nombre original :", archivo_origen.name)
print("Nombre canónico :", nombre_canonico)


df_can["Nombre_de_archivo"] = nombre_canonico

Nombre original : SC202511_Ok Sura.xlsx
Nombre canónico : Facturacion_Cesantia_202511.xlsx


🧹 Bloque 9 – Limpieza de filas de cierre (TOTAL o separadores)

En este bloque se eliminan las filas innecesarias al final del Excel, como totales o separadores vacíos:
- Se detectan filas con alto porcentaje de valores nulos.
- Se comparan sumas para confirmar si corresponde a una fila de “TOTAL”.
- Se eliminan automáticamente para evitar duplicar datos.

El resultado es un DataFrame limpio y sin filas redundantes.

In [12]:
df_can = fun.drop_trailing_mostly_null(
    df_can,
    null_check_exclude=("Nombre_de_archivo",),
    also_exclude_money_cols=("Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA"),
    null_ratio_threshold=0.80,
    verbose=True
)


Fila índice 101644 → null_ratio=95.83%
Fila índice 101643 → null_ratio=91.67%
Fila índice 101642 → null_ratio=8.33%

🧹 Eliminando 2 fila(s) finales mayoritariamente nulas:

--- Fila eliminada (índice original 101644) ---
       foliocredito rutafiliado dvafiliado NombreAfiliado Plazo MontoBruto  \
101644                                                                       

       fecotorgamiento fechaPrimerVto FechaUltimoVto ValorCuota  ...  \
101644                                                           ...   

       Corredora FECHA_DE_VENTA FECHA_DE_BAJA Poliza Tasa Prima_Bruta_mensual  \
101644                  TOTALES                                     492627458   

                     IVA         Prima_Neta Diferencia_CCLA  \
101644  78654804.2184829  413972653.7815718            5087   

                       Nombre_de_archivo  
101644  Facturacion_Cesantia_202511.xlsx  

[1 rows x 29 columns]


--- Fila eliminada (índice original 101643) ---
       foliocredito rutafili

🧮 Bloque 10 – Tipado de columnas y normalización de fechas

En este bloque se asignan los tipos de datos correctos a cada columna y se normalizan las fechas al formato YYYYMMDD:
- Se convierten valores numéricos, decimales y fechas de Excel.
- Se ajustan los tipos (Int64, float64, string).
- Se truncan las cadenas a la longitud máxima esperada por la base de datos.

Este paso asegura que los datos sean compatibles con el esquema de SQL Server

In [13]:
INT_COLS    = ["rutafiliado", "Plazo", "MontoBruto", "fecotorgamiento","fechaPrimerVto", "FechaUltimoVto", "ValorCuota","FechaPrima",
                "Prima", "Desgravamen", "FechaDefuncion", "FechaOrigen", "Poliza"]
BIGINT_COLS = ["foliocredito", "FolioOrigen"]
FLOAT_COLS  = ["TasaOrigen", "Tasa", "Prima_Bruta_mensual", "IVA", "Prima_Neta", "Diferencia_CCLA"]
STR_COLS    = ["NombreAfiliado", "OrigenDefuncion", "Producto", "Corredora", "Nombre_de_archivo"]
DV_COL      = "dvafiliado"
DATE_COLS   = ["FECHA_DE_VENTA","FECHA_DE_BAJA"]


# --- 0) Primero: convertir fechas a YYYYMMDD (Int64) ---
for c in DATE_COLS:
    if c in df_can.columns:
        df_can[c] = fun._parse_to_yyyymmdd_int(df_can[c])

# BIGINT -> pandas Int64 (acepta NaN)
for c in BIGINT_COLS:
    if c in df_can.columns:
        df_can[c] = fun.to_num_series(df_can[c]).astype("Int64")

# INT -> pandas Int64
for c in INT_COLS:
    if c in df_can.columns:
        df_can[c] = fun.to_num_series(df_can[c]).astype("Int64")

# FLOAT -> float64
for c in FLOAT_COLS:
    if c in df_can.columns:
        df_can[c] = fun.to_num_series(df_can[c]).astype("float64")

# DV: char(1) en mayúscula
if DV_COL in df_can.columns:
    df_can[DV_COL] = (
        df_can[DV_COL]
        .astype("string")
        .str.strip()
        .str.upper()
        .map(lambda x: x[:1] if pd.notna(x) and len(x) > 0 else pd.NA)
    )

if "NombreAfiliado" in df_can.columns:
    df_can["NombreAfiliado"] = df_can["NombreAfiliado"].astype("string").str.strip().str.slice(0, 50)

if "OrigenDefuncion" in df_can.columns:
    df_can["OrigenDefuncion"] = df_can["OrigenDefuncion"].astype("string").str.strip().str.slice(0, 50)

if "Producto" in df_can.columns:
    df_can["Producto"] = df_can["Producto"].astype("string").str.strip().str.slice(0, 50)

if "Corredora" in df_can.columns:
    df_can["Corredora"] = df_can["Corredora"].astype("string").str.strip().str.slice(0, 50)

if "Nombre_de_archivo" in df_can.columns:
    df_can["Nombre_de_archivo"] = df_can["Nombre_de_archivo"].astype("string").str.strip().str.slice(0, 50)

print("✅ dtypes DESPUÉS:\n", df_can.dtypes)

# --- 3) (Opcional) Validación rápida de nulos en columnas críticas ---
criticas = ["foliocredito","rutafiliado","dvafiliado","Nombre_de_archivo"]
presentes = [c for c in criticas if c in df_can.columns]
if presentes:
    print("\n🔎 Nulos en columnas críticas:")
    for c in presentes:
        print(f" - {c}: {int(df_can[c].isna().sum())} nulos")

cols_sql = [
    "foliocredito","rutafiliado","dvafiliado","NombreAfiliado","Plazo","MontoBruto","fecotorgamiento",
    "fechaPrimerVto","FechaUltimoVto","ValorCuota","FechaPrima","Prima","Desgravamen","FechaDefuncion",
    "OrigenDefuncion","Producto","FolioOrigen", "FechaOrigen","TasaOrigen", "Corredora", "FECHA_DE_VENTA","FECHA_DE_BAJA",
    "Poliza","Tasa","Prima_Bruta_mensual","IVA","Prima_Neta","Diferencia_CCLA","Nombre_de_archivo"
]

df_sql = df_can[[c for c in cols_sql if c in df_can.columns]].copy()

print("\n📦 df_sql listo con columnas:", list(df_sql.columns))

C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py:3170: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  d = pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)
C:\Users\aepmvalenzuela\traspaso-innominados\functions\functions.py:3170: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  d = pd.to_datetime(s, errors="coerce", dayfirst=False, infer_datetime_format=True)


✅ dtypes DESPUÉS:
 foliocredito                    Int64
rutafiliado                     Int64
dvafiliado                     object
NombreAfiliado         string[python]
Plazo                           Int64
MontoBruto                      Int64
fecotorgamiento                 Int64
fechaPrimerVto                  Int64
FechaUltimoVto                  Int64
ValorCuota                      Int64
FechaPrima                      Int64
Prima                           Int64
Desgravamen                     Int64
FechaDefuncion                  Int64
OrigenDefuncion        string[python]
Producto               string[python]
FolioOrigen                     Int64
FechaOrigen                     Int64
TasaOrigen                    float64
Corredora              string[python]
FECHA_DE_VENTA                  Int64
FECHA_DE_BAJA                   Int64
Poliza                          Int64
Tasa                          float64
Prima_Bruta_mensual           float64
IVA                           f

🔎 Bloque 11 – Validación de NOMBRE_ARCHIVO y conteo de filas por archivo

Este bloque valida:
- Que NOMBRE_ARCHIVO exista.
- Que tenga valores válidos (no vacíos).
- Cuenta cuántas filas trae cada archivo en el DataFrame.

Genera un diccionario {archivo → cantidad de filas}, usado luego para control de versiones.

In [14]:
assert "Nombre_de_archivo" in df_sql.columns, "Falta la columna 'Nombre_de_archivo' en df_sql."

df_sql["Nombre_de_archivo"] = (
    df_sql["Nombre_de_archivo"]
    .astype("string")
    .str.strip()
)

vals = [
    v for v in df_sql["Nombre_de_archivo"].dropna().unique()
    if str(v).strip() != ""
]

if not vals:
    raise SystemExit("🚨 No se encontraron valores válidos en 'Nombre_de_archivo'.")

counts = (
    df_sql["Nombre_de_archivo"]
    .value_counts(dropna=True)
    .to_dict()
)

print(f"📄 Se detectaron {len(vals)} Nombre_de_archivo distintos en el df_sql:")
for nombre, cnt in counts.items():
    print(f"   - {nombre}: {cnt} filas")

📄 Se detectaron 1 Nombre_de_archivo distintos en el df_sql:
   - Facturacion_Cesantia_202511.xlsx: 101643 filas


🚀 Bloque 12 – Carga con reemplazo selectivo por archivo (ETL transaccional)

Este bloque implementa el core del proceso ETL:
- Se ejecuta una sola transacción.
- Para cada NOMBRE_ARCHIVO:
- Consulta si existe data previa en SQL.
- Si existe → elimina solo ese archivo (reemplazo controlado).
- Inserta solo las filas nuevas.
- Imprime un resumen:
    - Filas reemplazadas
    - Filas insertadas
    - Archivos nuevos vs antiguos

Es un flujo robusto tipo "upsert por archivo".

In [17]:
resumen = []

# Nombre lógico de tabla
table_name = data["tablas_remotas"]["marsh_acumulado_r_bkp"]

for nombre_archivo in vals:

    # ------------------------------------------------------------
    # 1) Subconjunto del DataFrame
    # ------------------------------------------------------------
    df_sub = df_sql[df_sql["Nombre_de_archivo"] == nombre_archivo]

    if df_sub.empty:
        print(
            f"⚠️ No se encontraron filas en df_sql para "
            f"Nombre_de_archivo = '{nombre_archivo}'. Se omite."
        )
        continue

    # ------------------------------------------------------------
    # 2) Conteo previo en destino (COUNT *)
    # ------------------------------------------------------------
    sql_count = f"""
        SELECT COUNT(*) AS cnt
        FROM {schema}.{table_name}
        WHERE Nombre_de_archivo = '{nombre_archivo}'
    """

    df_cnt = fun.query_to_df(
        sql=sql_count,
        connection_string=connection_string,
        engine="pandas",
        return_iter=False,
    )

    existing_count = int(df_cnt.iloc[0, 0]) if not df_cnt.empty else 0

    # ------------------------------------------------------------
    # 3) Decisión: reemplazo vs inserción nueva
    # ------------------------------------------------------------
    if existing_count > 0:
        print(
            f"♻️ Se encontró data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}' ({existing_count} filas en {table_name})."
        )
        print("🧹 Eliminando filas anteriores para reemplazarlas...")

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="replace_partition",
            partition_column="Nombre_de_archivo",
            partition_values=[nombre_archivo],
            engine="pandas",
        )

        accion = "reemplazo"
        filas_borradas = summary["rows_deleted"]

        print(
            f"✅ Filas eliminadas en destino para "
            f"'{nombre_archivo}': {filas_borradas}"
        )

    else:
        print(
            f"🆕 No hay data previa para Nombre_de_archivo = "
            f"'{nombre_archivo}'. Se insertará como archivo nuevo."
        )

        summary = fun.df_to_db(
            df=df_sub,
            connection_string=connection_string,
            schema=schema,
            table=table_name,
            mode="append",
            engine="pandas",
        )

        accion = "inserción nueva"

    # ------------------------------------------------------------
    # 4) Log y resumen
    # ------------------------------------------------------------
    filas_sub = len(df_sub)
    print(
        f"📥 Insertadas {filas_sub} filas para "
        f"Nombre_de_archivo = '{nombre_archivo}'."
    )

    resumen.append(
        (nombre_archivo, filas_sub, existing_count, accion)
    )

# ------------------------------------------------------------
# 5) Resumen final (idéntico al original)
# ------------------------------------------------------------
print("\n📊 Resumen de carga por Nombre_de_archivo:")
for nombre_archivo, filas_sub, existing_count, accion in resumen:
    if accion == "reemplazo":
        print(
            f"   - {nombre_archivo}: {filas_sub} filas cargadas "
            f"(reemplazando {existing_count} previas)."
        )
    else:
        print(
            f"   - {nombre_archivo}: {filas_sub} filas cargadas "
            f"(archivo nuevo)."
        )

ProgrammingError: ('42S02', "[42S02] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Invalid object name 'dbo.MARSH_ACUMULADO_R_BKP'. (208) (SQLExecDirectW)")

🗑️ Bloque 13 – Eliminación del archivo procesado

Finalmente:
- Una vez cargado en SQL, el archivo fuente se elimina automáticamente.
- Maneja errores comunes:
    - Archivo en uso por Excel/OneDrive
    - Permisos insuficientes
    - Casos en que el archivo no existe
    
Cierra el ciclo ETL manteniendo la carpeta siempre limpia.

In [ ]:
try:
    if archivo_origen.exists() and archivo_origen.is_file():
        archivo_origen.unlink()
        print(f"\n🗑️ Archivo eliminado correctamente: {archivo_origen}")
    else:
        print(f"\n⚠️ No se pudo borrar el archivo porque no es un archivo válido: {archivo_origen}")
except PermissionError:
    print(f"\n⚠️ No se pudo borrar '{archivo_origen}': está en uso por OneDrive o Excel.")
except Exception as e:
    print(f"\n⚠️ Error inesperado al borrar archivo '{archivo_origen}': {e}")


🗑️ Archivo eliminado correctamente: C:\Users\aepnlizama\OneDrive - Seguros Suramericana, S.A\Escritorio\Programacion\Automatizaciones\Pruebas\CCLA\MARSH\SC202508_Ok Sura.xlsx


# Sección SQL

## VALIDACIONES / INSPECCIÓN

In [ ]:

query_1 = f"""
-- [MARSH] Muestra rápida
SELECT TOP 10 
    * 
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]};
"""

df_q1 = fun.query_to_df(
    sql=query_1,
    connection_string=connection_string,
    engine="pandas"
)

fun.pretty_table(df_q1)

In [ ]:
query_2 = f"""
-- [MARSH] Archivos cargados (desc)
SELECT DISTINCT 
    Nombre_de_archivo
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]}
ORDER BY 
    Nombre_de_archivo DESC;
"""

df_q2 = fun.query_to_df(
    sql=query_2,
    connection_string=connection_string,
    engine="pandas"
)

fun.pretty_table(df_q2)

## CONSTRUCCIÓN DE BASES FINALES POR CORREDOR / FUENTE

In [ ]:
query_3 = f"""
DROP TABLE IF EXISTS {data["tablas_remotas"]["marsh_final_acumulado_bkp"]};
"""

fun.exec_sql(query_3, connection_string)

In [ ]:
query_4 = f"""
SELECT *,
       /* 7561166 AS POLIZA, -- comentado en tu script */
       SUBSTRING(Nombre_de_archivo,
                 LEN(Nombre_de_archivo) - CHARINDEX('.', REVERSE(Nombre_de_archivo)) - 5,
                 6) AS MES_Recaudacion,
       8285 AS PLAN_TECNICO,
       4 AS PLAZO_CUOTAS,
       'Credito Consumo' AS Negocio
INTO {data["tablas_remotas"]["marsh_final_acumulado_bkp"]}
FROM {data["tablas_remotas"]["marsh_acumulado_r_bkp"]};
"""

fun.exec_sql(query_4, connection_string)

In [ ]:
query_5 = f"""
-- Ajuste provisorio: partner atrasado ~2 meses entre recaudación y real
UPDATE {data["tablas_remotas"]["marsh_final_acumulado_bkp"]}
SET 
    MES_Recaudacion = CASE
                        WHEN RIGHT(CAST(MES_Recaudacion AS VARCHAR(6)), 2) = '01' THEN (MES_Recaudacion - 89) -- enero -> año-1 mes12
                        ELSE (MES_Recaudacion - 1)
                      END;
"""

fun.exec_sql(query_5, connection_string)